In [8]:
# Feature Pipeline: Unwetterwarnung mit Open-Meteo API

In [9]:
# Hopsworks-Projektverbindungsskript
import os
from pathlib import Path

import hopsworks
from dotenv import load_dotenv

# Sicheres Laden der .env-Datei aus dem Projektstamm
project_root = Path.cwd().resolve()
if not (project_root / ".env").exists():
    project_root = project_root.parent

load_dotenv(project_root / ".env")

# Umgebungsvariablen abrufen
api_key = os.getenv("HOPSWORKS_API_KEY")
project_name = os.getenv("HOPSWORKS_PROJECT_NAME")

if not api_key or not project_name:
    raise ValueError(
        "HOPSWORKS_API_KEY und HOPSWORKS_PROJECT_NAME müssen in der .env-Datei gesetzt sein."
    )

# Verbindung zum Hopsworks-Projekt herstellen
project = hopsworks.login(
    api_key_value=api_key,
    project=project_name,
    host="eu-west.cloud.hopsworks.ai",
    port=443,
)

print(f"✅ Erfolgreich verbunden mit Projekt: {project.name}")

2026-09-23 19:43:44,561 INFO: Closing external client and cleaning up certificates.
2026-09-23 19:43:44,563 INFO: Connection closed.
2026-09-23 19:43:44,564 INFO: Initializing external client
2026-09-23 19:43:44,568 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-09-23 19:43:45,890 INFO: Python Engine initialized.



Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/44167
✅ Erfolgreich verbunden mit Projekt: fhnw_p1_weather_forcasts


In [10]:
# Rohdaten via API abrufen
import openmeteo_requests
import requests_cache
import pandas as pd
from retry_requests import retry

url = "https://api.open-meteo.com/v1/forecast"


def fetch_weather_data(latitude: float, longitude: float, past_days: int = 30, forecast_days: int = 1) -> pd.DataFrame:
    cache_session = requests_cache.CachedSession('.cache', expire_after=3600)
    retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
    openmeteo = openmeteo_requests.Client(session=retry_session)

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": [
            "temperature_2m",
            "relative_humidity_2m",
            "precipitation",
            "pressure_msl",
            "surface_pressure",
            "cloud_cover",
            "wind_speed_10m",
            "wind_gusts_10m",
            "cape",
        ],
        "past_days": past_days,
        "forecast_days": forecast_days,
        "timezone": "Europe/Berlin",
    }

    response = openmeteo.weather_api(url, params=params)[0]
    hourly = response.Hourly()
    start_ts = hourly.Time()
    num_points = len(hourly.Variables(0).ValuesAsNumpy())
    start_dt = pd.to_datetime(start_ts, unit="s", utc=True).tz_convert("Europe/Berlin")
    dates = pd.date_range(start=start_dt, periods=num_points, freq="h")

    df = pd.DataFrame({
        "time": dates,
        "temperature_2m": hourly.Variables(0).ValuesAsNumpy(),
        "relative_humidity_2m": hourly.Variables(1).ValuesAsNumpy(),
        "precipitation": hourly.Variables(2).ValuesAsNumpy(),
        "pressure_msl": hourly.Variables(3).ValuesAsNumpy(),
        "surface_pressure": hourly.Variables(4).ValuesAsNumpy(),
        "cloud_cover": hourly.Variables(5).ValuesAsNumpy(),
        "wind_speed_10m": hourly.Variables(6).ValuesAsNumpy(),
        "wind_gusts_10m": hourly.Variables(7).ValuesAsNumpy(),
        "cape": hourly.Variables(8).ValuesAsNumpy(),
    })

    df["latitude"] = latitude
    df["longitude"] = longitude
    df["location_id"] = f"{latitude}_{longitude}"
    return df


def clean_weather_data(df: pd.DataFrame) -> pd.DataFrame:
    """Bereinigt Zeitduplikate, physikalisch ungültige Werte und fehlende Werte."""
    df = df.copy()
    df["time"] = pd.to_datetime(df["time"], errors="coerce")
    df = df.dropna(subset=["time", "location_id"])
    df = df.drop_duplicates(subset=["location_id", "time"], keep="last")

    valid_ranges = {
        "temperature_2m": (-90, 60),
        "relative_humidity_2m": (0, 100),
        "precipitation": (0, 1000),
        "pressure_msl": (850, 1100),
        "surface_pressure": (850, 1100),
        "cloud_cover": (0, 100),
        "wind_speed_10m": (0, 250),
        "wind_gusts_10m": (0, 350),
        "cape": (0, 10000),
    }
    numeric_columns = list(valid_ranges)
    for column, (lower, upper) in valid_ranges.items():
        values = pd.to_numeric(df[column], errors="coerce")
        df[column] = values.mask((values < lower) | (values > upper))

    df = df.sort_values(["location_id", "time"])
    for column in numeric_columns:
        df[column] = df.groupby("location_id")[column].transform(
            lambda values: values.interpolate(limit_direction="both")
        )
        df[column] = df[column].fillna(
            df.groupby("location_id")[column].transform("median")
        )

    required_columns = [
        "temperature_2m", "relative_humidity_2m", "precipitation",
        "pressure_msl", "surface_pressure", "cloud_cover",
        "wind_speed_10m", "wind_gusts_10m", "cape",
    ]
    return df.dropna(subset=required_columns).reset_index(drop=True)


locations = [
    {"name": "Muenchen", "lat": 48.1351, "lon": 11.5820},
    {"name": "Hamburg", "lat": 53.5511, "lon": 9.9937},
]

raw_dfs = []
for loc in locations:
    df = fetch_weather_data(loc["lat"], loc["lon"])
    df["location_name"] = loc["name"]
    raw_dfs.append(df)

raw_df = clean_weather_data(pd.concat(raw_dfs, ignore_index=True))
print(f"✅ {len(raw_df)} bereinigte Zeilen Rohdaten abgerufen")
print(raw_df.head())

✅ 1488 bereinigte Zeilen Rohdaten abgerufen
                       time  temperature_2m  relative_humidity_2m  \
0 2026-08-24 00:00:00+02:00       15.665000                  72.0   
1 2026-08-24 01:00:00+02:00       14.964999                  71.0   
2 2026-08-24 02:00:00+02:00       14.014999                  78.0   
3 2026-08-24 03:00:00+02:00       13.514999                  80.0   
4 2026-08-24 04:00:00+02:00       12.865000                  83.0   

   precipitation  pressure_msl  surface_pressure  cloud_cover  wind_speed_10m  \
0            0.0   1021.900024        961.169556        100.0        3.396233   
1            0.0   1022.099976        961.215393        100.0        5.001280   
2            0.0   1021.700012        960.644958         75.0        5.506941   
3            0.0   1021.799988        960.636353          0.0        3.096837   
4            0.0   1021.700012        960.408447         22.0        3.415260   

   wind_gusts_10m  cape  latitude  longitude     locat

In [11]:
# Features engineeren (Rolling Windows, Aggregationen)
# Der Zielhorizont beträgt drei Stunden.
TARGET_HORIZON_HOURS = 3


def engineer_features(df):
    """Erstellt Wetterfeatures und ein drei Stunden vorausblickendes Unwetter-Label."""
    df = df.sort_values(["location_id", "time"]).copy()
    if "wind_gusts_10m" not in df.columns:
        raise ValueError("wind_gusts_10m muss direkt aus der Wetter-API vorhanden sein.")

    df["pressure_msl"] = pd.to_numeric(df["pressure_msl"], errors="coerce")
    df["wind_gusts_10m"] = pd.to_numeric(df["wind_gusts_10m"], errors="coerce")
    df["cape"] = pd.to_numeric(df["cape"], errors="coerce")

    grouped = df.groupby("location_id")
    for window in [3, 6, 12]:
        df[f"precip_rolling_sum_{window}h"] = grouped["precipitation"].transform(
            lambda values: values.rolling(window, min_periods=1).sum()
        )
        df[f"wind_gust_max_{window}h"] = grouped["wind_gusts_10m"].transform(
            lambda values: values.rolling(window, min_periods=1).max()
        )
        df[f"pressure_mean_{window}h"] = grouped["pressure_msl"].transform(
            lambda values: values.rolling(window, min_periods=1).mean()
        )

    df["pressure_change_3h"] = grouped["pressure_msl"].transform(lambda values: values.diff(3))
    df["pressure_drop_rate"] = df["pressure_change_3h"] / 3
    df["wind_gust_anomaly"] = grouped["wind_gusts_10m"].transform(
        lambda values: values - values.rolling(24, min_periods=1).mean()
    )
    df["temp_change_3h"] = grouped["temperature_2m"].transform(
        lambda values: values.diff(3)
    )
    df["cape_risk_level"] = pd.cut(
        df["cape"], bins=[0, 500, 1500, 3000, float("inf")],
        labels=["niedrig", "moderat", "hoch", "extrem"]
    )

    # Das Ereignis wird aus Wetterwerten der Zielstunde abgeleitet.
    storm_event_now = (
        (df["wind_gusts_10m"] > 40)
        | (df["precip_rolling_sum_3h"] > 10)
        | (df["pressure_drop_rate"] < -0.75)
    ).astype(int)

    # Drei-Stunden-Frühwarnziel: Das Ereignislabel wird je Standort
    # um drei Stunden zurückverschoben.
    df["is_severe_weather"] = (
        storm_event_now.groupby(df["location_id"]).shift(-TARGET_HORIZON_HOURS)
    )
    df = df.dropna(subset=["is_severe_weather"]).copy()
    df["is_severe_weather"] = df["is_severe_weather"].astype(int)

    return df

feature_df = engineer_features(raw_df)
print(feature_df.head())
print(f"✅ Features erstellt: {feature_df.shape[1]} Spalten")
print(f"✅ Labelverteilung: {feature_df['is_severe_weather'].value_counts().to_dict()}")

                       time  temperature_2m  relative_humidity_2m  \
0 2026-08-24 00:00:00+02:00       15.665000                  72.0   
1 2026-08-24 01:00:00+02:00       14.964999                  71.0   
2 2026-08-24 02:00:00+02:00       14.014999                  78.0   
3 2026-08-24 03:00:00+02:00       13.514999                  80.0   
4 2026-08-24 04:00:00+02:00       12.865000                  83.0   

   precipitation  pressure_msl  surface_pressure  cloud_cover  wind_speed_10m  \
0            0.0   1021.900024        961.169556        100.0        3.396233   
1            0.0   1022.099976        961.215393        100.0        5.001280   
2            0.0   1021.700012        960.644958         75.0        5.506941   
3            0.0   1021.799988        960.636353          0.0        3.096837   
4            0.0   1021.700012        960.408447         22.0        3.415260   

   wind_gusts_10m  cape  ...  pressure_mean_6h  precip_rolling_sum_12h  \
0       11.520000   0.0 

In [12]:
# Dataframe mit Features und Label erstellen
def build_final_dataframe(df):
    """
    Selektiert relevante Spalten und bereitet Primary Key + Event Time vor.
    """
    desired_columns = [
        "location_id",
        "location_name",
        "latitude",
        "longitude",
        "time",
        "temperature_2m",
        "relative_humidity_2m",
        "precipitation",
        "pressure_msl",
        "surface_pressure",
        "cloud_cover",
        "wind_speed_10m",
        "wind_gusts_10m",
        "cape",
        "precip_rolling_sum_3h",
        "precip_rolling_sum_6h",
        "precip_rolling_sum_12h",
        "wind_gust_max_3h",
        "wind_gust_max_6h",
        "wind_gust_max_12h",
        "pressure_mean_3h",
        "pressure_mean_6h",
        "pressure_mean_12h",
        "pressure_change_3h",
        "pressure_drop_rate",
        "wind_gust_anomaly",
        "temp_change_3h",
        "is_severe_weather",
    ]

    feature_columns = [col for col in desired_columns if col in df.columns]
    final_df = df[feature_columns].copy()

    # Hopsworks erwartet fuer cape das Double-Schema der bestehenden Feature Group.
    final_df["cape"] = pd.to_numeric(final_df["cape"], errors="coerce").astype("float64")

    # Primary Key als eindeutige ID
    final_df["event_id"] = (
        final_df["location_id"] + "_" +
        final_df["time"].dt.strftime("%Y%m%d%H%M")
    )

    # Event Time sicherstellen
    final_df["event_time"] = final_df["time"]

    # NaN-Werte behandeln (z.B. am Anfang der Rolling Windows)
    final_df = final_df.dropna(subset=["pressure_drop_rate", "wind_gust_anomaly"]).copy()

    return final_df

final_df = build_final_dataframe(feature_df)
print(f"✅ Finaler DataFrame: {final_df.shape}")
print(final_df.head())

✅ Finaler DataFrame: (1476, 30)
      location_id location_name  latitude  longitude  \
3  48.1351_11.582      Muenchen   48.1351     11.582   
4  48.1351_11.582      Muenchen   48.1351     11.582   
5  48.1351_11.582      Muenchen   48.1351     11.582   
6  48.1351_11.582      Muenchen   48.1351     11.582   
7  48.1351_11.582      Muenchen   48.1351     11.582   

                       time  temperature_2m  relative_humidity_2m  \
3 2026-08-24 03:00:00+02:00       13.514999                  80.0   
4 2026-08-24 04:00:00+02:00       12.865000                  83.0   
5 2026-08-24 05:00:00+02:00       12.264999                  84.0   
6 2026-08-24 06:00:00+02:00       11.865000                  84.0   
7 2026-08-24 07:00:00+02:00       11.565000                  84.0   

   precipitation  pressure_msl  surface_pressure  ...  pressure_mean_3h  \
3            0.0   1021.799988        960.636353  ...       1021.866659   
4            0.0   1021.700012        960.408447  ...       1021.7

In [13]:
# Feature Group im Hopsworks Feature Store erstellen und Daten einfügen

# Access the feature store
fs = project.get_feature_store()

# Feature Group
feature_group = fs.get_or_create_feature_group(
    name="weather_features_batch",
    version=1,
    primary_key=["event_id"],
    event_time="event_time",
    time_travel_format="HUDI",
    stream=True,
    description="Stündliche Wetterfeatures für die Unwetterprognose",
)

feature_group.stream = True
feature_group.insert(final_df, write_options={"wait_for_job": True})
print(f"✅ {len(final_df)} Zeilen in weather_features_batch geschrieben")

Uploading Dataframe: 100.00% |██████████| Rows 1476/1476 | Elapsed Time: 00:00 | Remaining Time: 00:00


Launching job: weather_features_batch_1_offline_fg_materialization
Job started successfully, you can follow the progress at 
https://eu-west.cloud.hopsworks.ai:443/p/44167/jobs/named/weather_features_batch_1_offline_fg_materialization/executions


2026-09-23 19:44:10,964 INFO: Waiting for execution to finish. Current state: SUBMITTED. Final status: UNDEFINED
2026-09-23 19:44:14,209 INFO: Waiting for execution to finish. Current state: RUNNING. Final status: UNDEFINED
2026-09-23 19:47:36,237 INFO: Waiting for execution to finish. Current state: FINISHED. Final status: SUCCEEDED
2026-09-23 19:47:36,644 INFO: Waiting for log aggregation to finish.
2026-09-23 19:47:36,645 INFO: Execution finished successfully.


✅ 1476 Zeilen in weather_features_batch geschrieben


In [14]:
# Feature Pipeline
def run_feature_pipeline(locations, fs=None, past_days=30, forecast_days=3, upload=False):
    """Erstellt Wetterfeatures und optional einen Hopsworks-Upload.

    forecast_days=3 bestimmt nur den Datenabruf für drei Prognosetage;
    der Modell-Zielhorizont wird separat mit TARGET_HORIZON_HOURS festgelegt.
    """
    raw_dfs = []
    for loc in locations:
        df = fetch_weather_data(loc["lat"], loc["lon"], past_days, forecast_days)
        df["location_name"] = loc["name"]
        raw_dfs.append(df)

    raw_df = clean_weather_data(pd.concat(raw_dfs, ignore_index=True))
    feature_df = engineer_features(raw_df)
    final_df = build_final_dataframe(feature_df)

    weather_fg = None
    if upload:
        if fs is None:
            raise ValueError("Für upload=True muss ein Hopsworks Feature Store übergeben werden.")

        weather_fg = fs.get_or_create_feature_group(
            name="weather_features_batch",
            version=1,
            primary_key=["event_id"],
            event_time="event_time",
            time_travel_format="HUDI",
            stream=True,
            description="Stündliche Wetterfeatures für die Unwetterprognose",
        )
        weather_fg.stream = True
        weather_fg.insert(final_df, write_options={"wait_for_job": True})
        print(f"✅ {len(final_df)} Zeilen in weather_features_batch geschrieben")

    return final_df, weather_fg

# Erneuter Upload mit 30 Tagen Historie für beide Labelklassen.
final_df, weather_fg = run_feature_pipeline(locations, fs, past_days=30, upload=True)
print(f"✅ End-to-End-Pipeline erfolgreich: {final_df.shape}")
print(f"✅ Labelverteilung im Upload: {final_df['is_severe_weather'].value_counts().to_dict()}")

Uploading Dataframe: 100.00% |██████████| Rows 1572/1572 | Elapsed Time: 00:00 | Remaining Time: 00:00


Launching job: weather_features_batch_1_offline_fg_materialization
Job started successfully, you can follow the progress at 
https://eu-west.cloud.hopsworks.ai:443/p/44167/jobs/named/weather_features_batch_1_offline_fg_materialization/executions


2026-09-23 19:47:54,475 INFO: Waiting for execution to finish. Current state: SUBMITTED. Final status: UNDEFINED
2026-09-23 19:47:57,724 INFO: Waiting for execution to finish. Current state: RUNNING. Final status: UNDEFINED
2026-09-23 19:51:16,529 INFO: Waiting for execution to finish. Current state: SUCCEEDING. Final status: UNDEFINED
2026-09-23 19:51:23,115 INFO: Waiting for execution to finish. Current state: FINISHED. Final status: SUCCEEDED
2026-09-23 19:51:23,566 INFO: Waiting for log aggregation to finish.
2026-09-23 19:51:23,566 INFO: Execution finished successfully.


✅ 1572 Zeilen in weather_features_batch geschrieben
✅ End-to-End-Pipeline erfolgreich: (1572, 30)
✅ Labelverteilung im Upload: {0: 1451, 1: 121}
